# 01 - YOLO26n / YOLO11n / YOLO12n (Ultralytics)

Trois modeles x 5 folds. Augmentation = `augment.AUG` appliquee telle quelle ; tous les autres hyperparametres restent les defauts Ultralytics. Les metriques viennent de l'evaluateur COCO unifie, pas de `model.val()`.

In [ ]:
# --- Installation ---
!pip install -q ultralytics wandb pycocotools

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
import os, sys
REPO_DIR = "/content/aphids_detection"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/EmmaDub/aphids_detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__)

In [ ]:
# --- Configuration ---
# Les chemins par defaut sont ceux de aphids_det/config.py. Pour les changer,
# decommenter et adapter, puis relancer cfg.refresh().
from pathlib import Path
import aphids_det.config as cfg

# cfg.BASE_DIR  = Path("/content/drive/MyDrive/.../tuile_viz02_640_128")
# cfg.OUT_DIR   = Path("/content/drive/MyDrive/.../puceron_model_article/data")
# cfg.EPOCHS    = 30
# cfg.USE_WANDB = True

cfg.refresh()
cfg.summary()

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Suivi W&B (facultatif : mettre cfg.USE_WANDB = False pour s'en passer) ---
if cfg.USE_WANDB:
    import wandb
    wandb.login()
    os.environ["WANDB_PROJECT"] = cfg.WANDB_PROJECT
    print("W&B -> projet", cfg.WANDB_PROJECT)

## Comparaison en validation croisee

Reprise automatique : un (modele, fold) deja present dans le CSV est saute.

In [ ]:
from aphids_det.runners import ultralytics_runner
df = ultralytics_runner.run_cv()      # folds 0-4 pour les 3 modeles

In [ ]:
# --- Etat du CSV de benchmark ---
import pandas as pd
d = pd.read_csv(cfg.CSV_CV)
print(d.groupby("modele")["fold"].count().to_string(), "\n")
d[["modele", "fold", "map50_macro", "map5095_macro", "latency_cpu_ms",
   "n_params_M", "train_time_s"]].tail(10)